In [1]:
# Install Dependencies
!pip install sentence-transformers==3.4.1
!pip install faiss-cpu==1.10.0
!pip install chromadb==0.6.3
!pip install groq
!pip install numpy==1.26.4
!pip install tqdm==4.67.1

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [2]:
import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

In [3]:
# Imports
import os
import numpy as np
from tqdm import tqdm

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# FAISS
import faiss

# ChromaDB
import chromadb
from chromadb.config import Settings

from groq import Groq

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [4]:
# Sample Documents
documents = [
    # AI / ML
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
    "Deep learning uses neural networks with many layers to model complex patterns in data.",
    "Natural language processing allows computers to understand and generate human language.",
    "Transformers are a neural network architecture that revolutionized NLP through self-attention.",
    "Reinforcement learning trains agents by rewarding desirable actions in an environment.",

    # Python
    "Python is a high-level programming language known for its simplicity and readability.",
    "NumPy provides support for large multi-dimensional arrays and mathematical operations.",
    "Pandas is a data manipulation library built on top of NumPy for structured data.",
    "FastAPI is a modern web framework for building APIs with Python using type hints.",
    "Virtual environments in Python isolate project dependencies from the system Python.",

    # Databases
    "PostgreSQL is a powerful open-source relational database management system.",
    "MongoDB is a NoSQL database that stores data in flexible JSON-like documents.",
    "Redis is an in-memory data structure store used as a database, cache, and message broker.",
    "Vector databases store high-dimensional embeddings and enable similarity search.",
    "ChromaDB is an open-source vector database designed for AI-powered applications.",

    # Cloud
    "Docker containers package applications with their dependencies for consistent deployment.",
    "Kubernetes orchestrates containerized applications across clusters of machines.",
    "AWS Lambda enables serverless computing by running code in response to events.",
    "CI/CD pipelines automate the building, testing, and deployment of software.",
    "Microservices architecture breaks applications into small, independently deployable services.",
]

print(f"✅ Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc[:60]}...")

✅ Loaded 20 documents
  [0] Machine learning is a subset of artificial intelligence that...
  [1] Deep learning uses neural networks with many layers to model...
  [2] Natural language processing allows computers to understand a...
  [3] Transformers are a neural network architecture that revoluti...
  [4] Reinforcement learning trains agents by rewarding desirable ...
  [5] Python is a high-level programming language known for its si...
  [6] NumPy provides support for large multi-dimensional arrays an...
  [7] Pandas is a data manipulation library built on top of NumPy ...
  [8] FastAPI is a modern web framework for building APIs with Pyt...
  [9] Virtual environments in Python isolate project dependencies ...
  [10] PostgreSQL is a powerful open-source relational database man...
  [11] MongoDB is a NoSQL database that stores data in flexible JSO...
  [12] Redis is an in-memory data structure store used as a databas...
  [13] Vector databases store high-dimensional embeddings and ena

In [5]:
#  Generate Embeddings
# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Model loaded: all-MiniLM-L6-v2")

# Generate embeddings for all documents

embeddings = model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings generated!")
print(f"   Shape : {embeddings.shape}")
print(f"   Dtype : {embeddings.dtype}")
print(f"   Sample vector (first 5 dims of doc[0]): {embeddings[0][:5]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings generated!
   Shape : (20, 384)
   Dtype : float32
   Sample vector (first 5 dims of doc[0]): [-0.02090065  0.02186695  0.03556452  0.02934006  0.04585582]


In [6]:
#  FAISS Index: Build & Search

# ── Build ──────────────────────────────────────────────
# Get embedding dimension
dimension = embeddings.shape[1]  # 384

# Create a flat L2 index (exact search, no approximation)
faiss_index = faiss.IndexFlatL2(dimension)

# FAISS requires float32
embeddings_f32 = embeddings.astype(np.float32)

# Add all document embeddings to the index
faiss_index.add(embeddings_f32)

print(f"✅ FAISS index built!")
print(f"   Vectors stored : {faiss_index.ntotal}")
print(f"   Dimension      : {dimension}")

# ── Search ─────────────────────────────────────────────
def search_faiss(query, top_k=3):
    # Embed the query
    query_vector = model.encode([query], convert_to_numpy=True).astype(np.float32)

    # Search — returns distances and indices
    distances, indices = faiss_index.search(query_vector, top_k)

    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        results.append({
            "rank"    : rank + 1,
            "index"   : int(idx),
            "document": documents[idx],
            "distance": round(float(dist), 4)   # L2 distance (lower = more similar)
        })
    return results

# ── Test ───────────────────────────────────────────────
query = "How do neural networks learn?"
print(f"\n🔍 Query: '{query}'")
print("-" * 60)

faiss_results = search_faiss(query, top_k=3)
for r in faiss_results:
    print(f"  Rank {r['rank']} | Distance: {r['distance']}")
    print(f"  → {r['document']}\n")

✅ FAISS index built!
   Vectors stored : 20
   Dimension      : 384

🔍 Query: 'How do neural networks learn?'
------------------------------------------------------------
  Rank 1 | Distance: 0.8928
  → Deep learning uses neural networks with many layers to model complex patterns in data.

  Rank 2 | Distance: 1.0658
  → Machine learning is a subset of artificial intelligence that enables systems to learn from data.

  Rank 3 | Distance: 1.1607
  → Transformers are a neural network architecture that revolutionized NLP through self-attention.



In [7]:
# ChromaDB Index: Build & Search


# Create an in-memory ChromaDB client (no disk needed in Colab)
chroma_client = chromadb.Client()

# Create a collection (like a table in a relational DB)
collection = chroma_client.create_collection(
    name="semantic_search",
    metadata={"hnsw:space": "cosine"}   # use cosine similarity
)

# Add all documents with their embeddings
collection.add(
    ids        = [f"doc_{i}" for i in range(len(documents))],   # unique IDs
    embeddings = embeddings.tolist(),                            # vectors
    documents  = documents                                       # raw text stored alongside
)

print(f"✅ ChromaDB collection built!")
print(f"   Documents stored : {collection.count()}")

# ── Search ─────────────────────────────────────────────
def search_chroma(query, top_k=3):
    # Embed the query
    query_vector = model.encode([query], convert_to_numpy=True).tolist()

    # Query the collection
    results = collection.query(
        query_embeddings = query_vector,
        n_results        = top_k
    )

    formatted = []
    for rank, (doc, dist) in enumerate(zip(
        results["documents"][0],
        results["distances"][0]
    )):
        formatted.append({
            "rank"      : rank + 1,
            "document"  : doc,
            "distance"  : round(dist, 4)   # cosine distance (lower = more similar)
        })
    return formatted

# ── Test ───────────────────────────────────────────────
query = "How do neural networks learn?"
print(f"\n🔍 Query: '{query}'")
print("-" * 60)

chroma_results = search_chroma(query, top_k=3)
for r in chroma_results:
    print(f"  Rank {r['rank']} | Distance: {r['distance']}")
    print(f"  → {r['document']}\n")

✅ ChromaDB collection built!
   Documents stored : 20

🔍 Query: 'How do neural networks learn?'
------------------------------------------------------------
  Rank 1 | Distance: 0.4464
  → Deep learning uses neural networks with many layers to model complex patterns in data.

  Rank 2 | Distance: 0.5329
  → Machine learning is a subset of artificial intelligence that enables systems to learn from data.

  Rank 3 | Distance: 0.5804
  → Transformers are a neural network architecture that revolutionized NLP through self-attention.



In [8]:
# Side-by-Side Comparison

def compare(query, top_k=3):
    print(f"\n{'='*65}")
    print(f"  🔍 Query: '{query}'")
    print(f"{'='*65}")

    f_results = search_faiss(query, top_k)
    c_results = search_chroma(query, top_k)

    print(f"\n{'FAISS (L2 distance)':<45} {'ChromaDB (cosine distance)'}")
    print(f"{'-'*45} {'-'*45}")

    for f, c in zip(f_results, c_results):
        print(f"\n  Rank {f['rank']}")
        print(f"  FAISS    [{f['distance']}] {f['document'][:70]}")
        print(f"  ChromaDB [{c['distance']}] {c['document'][:70]}")

    print()

# Run comparisons on different query types
compare("How do neural networks learn?")
compare("What is a vector database?")
compare("How to deploy Python applications?")
compare("What is reinforcement learning?")


  🔍 Query: 'How do neural networks learn?'

FAISS (L2 distance)                           ChromaDB (cosine distance)
--------------------------------------------- ---------------------------------------------

  Rank 1
  FAISS    [0.8928] Deep learning uses neural networks with many layers to model complex p
  ChromaDB [0.4464] Deep learning uses neural networks with many layers to model complex p

  Rank 2
  FAISS    [1.0658] Machine learning is a subset of artificial intelligence that enables s
  ChromaDB [0.5329] Machine learning is a subset of artificial intelligence that enables s

  Rank 3
  FAISS    [1.1607] Transformers are a neural network architecture that revolutionized NLP
  ChromaDB [0.5804] Transformers are a neural network architecture that revolutionized NLP


  🔍 Query: 'What is a vector database?'

FAISS (L2 distance)                           ChromaDB (cosine distance)
--------------------------------------------- ---------------------------------------------

  Ran

In [9]:
# ── Cell 8: Groq Answer Layer ───────────────────────────────────

from google.colab import userdata
from groq import Groq

# ── Initialize Groq client ──────────────────────────────────────
try:
    groq_client = Groq(api_key=userdata.get("GROQ_API"))
    print("✅ Groq client initialized!")
except Exception as e:
    groq_client = None
    print(f"❌ Groq init failed: {e}")


def ask_groq(query, top_k=3):
    """
    RAG pipeline:
    1. Retrieve top_k documents from ChromaDB
    2. Build a prompt with those docs as context
    3. Send to Groq LLM and return the answer
    """
    # Step 1: Retrieve top documents from ChromaDB
    results = search_chroma(query, top_k)
    context_docs = [r["document"] for r in results]

    # Step 2: Build the prompt
    context = "\n".join([f"- {doc}" for doc in context_docs])
    prompt = f"""You are a helpful assistant. Answer the user's question based only on the context provided below.
If the context doesn't contain enough information, say so honestly.

Context:
{context}

Question: {query}

Answer:"""

    # Step 3: Call Groq
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",  # free, fast, works in India
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            temperature=0.2                # low = factual answers
        )
        answer = response.choices[0].message.content.strip()

    except Exception as e:
        # Graceful fallback — pipeline never crashes
        answer = f"[Groq unavailable: {type(e).__name__}] Top match: {context_docs[0]}"

    return answer, context_docs


# ── Test ────────────────────────────────────────────────────────
query = "How do neural networks learn?"
print(f"🔍 Query: {query}\n")

answer, sources = ask_groq(query)

print("📄 Retrieved Context:")
for i, doc in enumerate(sources, 1):
    print(f"  [{i}] {doc}")

print(f"\n🤖 Groq Answer:\n{answer}")

✅ Groq client initialized!
🔍 Query: How do neural networks learn?

📄 Retrieved Context:
  [1] Deep learning uses neural networks with many layers to model complex patterns in data.
  [2] Machine learning is a subset of artificial intelligence that enables systems to learn from data.
  [3] Transformers are a neural network architecture that revolutionized NLP through self-attention.

🤖 Groq Answer:
Based on the provided context, it doesn't explicitly explain how neural networks learn. However, it does mention that deep learning uses neural networks with many layers to model complex patterns in data, and that machine learning is a subset of artificial intelligence that enables systems to learn from data.

While it doesn't directly answer the question, it implies that neural networks learn through the process of modeling complex patterns in data, which is a key concept in machine learning. However, for a more detailed explanation, additional context would be necessary.


In [10]:
# ── Cell 9: Full End-to-End Pipeline ────────────────────────────

class SemanticSearchEngine:
    def __init__(self, documents):
        print("🚀 Initializing Semantic Search Engine...")

        # Load embedding model
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.documents = documents

        # Generate embeddings
        print("   Generating embeddings...")
        self.embeddings = self.model.encode(
            documents,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # Build FAISS index
        print("   Building FAISS index...")
        dim = self.embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatL2(dim)
        self.faiss_index.add(self.embeddings.astype(np.float32))

        # Build ChromaDB collection
        print("   Building ChromaDB collection...")
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.create_collection(
            name="engine",
            metadata={"hnsw:space": "cosine"}
        )
        self.collection.add(
            ids        = [f"doc_{i}" for i in range(len(documents))],
            embeddings = self.embeddings.tolist(),
            documents  = documents
        )

        # Groq client (replaces Gemini)
        self.groq_client = groq_client  # initialized in Cell 8

        print("✅ Engine ready!\n")

    def search(self, query, top_k=3):
        q_vec = self.model.encode([query], convert_to_numpy=True).astype(np.float32)

        # FAISS
        distances, indices = self.faiss_index.search(q_vec, top_k)
        faiss_results = [
            {"rank": i+1, "document": self.documents[idx], "score": round(float(d), 4)}
            for i, (idx, d) in enumerate(zip(indices[0], distances[0]))
        ]

        # ChromaDB
        chroma_out = self.collection.query(query_embeddings=q_vec.tolist(), n_results=top_k)
        chroma_results = [
            {"rank": i+1, "document": doc, "score": round(d, 4)}
            for i, (doc, d) in enumerate(zip(chroma_out["documents"][0], chroma_out["distances"][0]))
        ]

        return faiss_results, chroma_results

    def answer(self, query, top_k=3):
        faiss_results, chroma_results = self.search(query, top_k)

        # Build context from ChromaDB top results
        context = "\n".join([f"- {r['document']}" for r in chroma_results])
        prompt = f"""You are a helpful assistant. Answer the question using only the context below.
If the context doesn't contain enough information, say so honestly.

Context:
{context}

Question: {query}
Answer:"""

        # Call Groq
        try:
            response = self.groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
                temperature=0.2
            )
            groq_answer = response.choices[0].message.content.strip()

        except Exception as e:
            groq_answer = f"[Groq unavailable: {type(e).__name__}] Top match: {chroma_results[0]['document']}"

        return {
            "query"         : query,
            "faiss_results" : faiss_results,
            "chroma_results": chroma_results,
            "gemini_answer" : groq_answer   # key name unchanged — print statements below still work
        }


# ── Run the engine ───────────────────────────────────────────────
engine = SemanticSearchEngine(documents)

queries = [
    "How do neural networks learn?",
    "What is a vector database?",
    "How to deploy applications with Docker?"
]

for q in queries:
    result = engine.answer(q)

    print(f"\n{'='*65}")
    print(f"🔍 {result['query']}")
    print(f"{'='*65}")

    print("\n📊 FAISS Results:")
    for r in result["faiss_results"]:
        print(f"  [{r['rank']}] ({r['score']}) {r['document'][:70]}")

    print("\n📊 ChromaDB Results:")
    for r in result["chroma_results"]:
        print(f"  [{r['rank']}] ({r['score']}) {r['document'][:70]}")

    print(f"\n🤖 Groq Answer:\n{result['gemini_answer']}")

🚀 Initializing Semantic Search Engine...
   Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   Building FAISS index...
   Building ChromaDB collection...
✅ Engine ready!


🔍 How do neural networks learn?

📊 FAISS Results:
  [1] (0.8928) Deep learning uses neural networks with many layers to model complex p
  [2] (1.0658) Machine learning is a subset of artificial intelligence that enables s
  [3] (1.1607) Transformers are a neural network architecture that revolutionized NLP

📊 ChromaDB Results:
  [1] (0.4464) Deep learning uses neural networks with many layers to model complex p
  [2] (0.5329) Machine learning is a subset of artificial intelligence that enables s
  [3] (0.5804) Transformers are a neural network architecture that revolutionized NLP

🤖 Groq Answer:
Unfortunately, the context doesn't contain enough information about how neural networks learn. It mentions that deep learning uses neural networks with many layers to model complex patterns in data, but it doesn't explain the learning process itself.

🔍 What is a vector database?

📊 FAISS Results:
  [1] (0.6922) Vec